# nuscenes.convertersデバッグ用可視化スクリプト

In [ ]:
# ExtractorとConverter実行
import sys
import os

sys.path.insert(0, os.path.abspath('..'))

from src.carla.carla_client import get_carla_client, set_world
from src.carla.map.nuscenes.create_nusc_map import extract_carla_map_data, convert_carla_map_to_nuscenes

HOST = 'localhost'
PORT = 2000
MAP_NAME = None
SAMPLING_RESOLUTION = 1.0

# CARLA接続
client = get_carla_client(host=HOST, port=PORT)
world = set_world(client, map_name=MAP_NAME)
map_name = world.get_map().name.split('/')[-1]

# Extractor実行
carla_map_data = extract_carla_map_data(world, sampling_resolution=SAMPLING_RESOLUTION)

# Converter実行
map_data, basemap_image = convert_carla_map_to_nuscenes(carla_map_data, merge_buffer=0.2)

In [ ]:
# Drivable Areaの可視化（ポリゴンごとに色分け）
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon

# ノードtoken→座標のマッピング
node_map = {n['token']: (n['x'], n['y']) for n in map_data['node']}
# ポリゴンtoken→外周座標のマッピング
polygon_map = {}
for p in map_data['polygon']:
    coords = [node_map[nt] for nt in p['exterior_node_tokens']]
    polygon_map[p['token']] = coords

fig, ax = plt.subplots(1, 1, figsize=(14, 14))

for da in map_data['drivable_area']:
    n_polys = len(da['polygon_tokens'])
    cmap = plt.cm.get_cmap('tab20', max(n_polys, 1))
    for i, pt in enumerate(da['polygon_tokens']):
        if pt not in polygon_map:
            continue
        coords = polygon_map[pt]
        color = cmap(i % 20)
        poly = MplPolygon(coords, closed=True, facecolor=color, edgecolor='black', linewidth=0.3, alpha=0.7)
        ax.add_patch(poly)

ax.set_aspect('equal')
ax.autoscale_view()
ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_title(f'Drivable Area ({sum(len(da["polygon_tokens"]) for da in map_data["drivable_area"])} polygons)')
plt.tight_layout()
plt.show()

In [ ]:
# Road Segmentの可視化（要素ごとに色分け）
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon

fig, ax = plt.subplots(1, 1, figsize=(14, 14))

# 非交差点をroad_segmentごとに色分け
road_segments = [rs for rs in map_data['road_segment'] if not rs['is_intersection']]
intersection_segments = [rs for rs in map_data['road_segment'] if rs['is_intersection']]

cmap_road = plt.cm.get_cmap('tab20', max(len(road_segments), 1))
cmap_int = plt.cm.get_cmap('Set3', max(len(intersection_segments), 1))

for i, rs in enumerate(road_segments):
    pt = rs['polygon_token']
    if pt not in polygon_map:
        continue
    coords = polygon_map[pt]
    color = cmap_road(i % 20)
    poly = MplPolygon(coords, closed=True, facecolor=color, edgecolor='black', linewidth=0.5, alpha=0.7)
    ax.add_patch(poly)

for i, rs in enumerate(intersection_segments):
    pt = rs['polygon_token']
    if pt not in polygon_map:
        continue
    coords = polygon_map[pt]
    color = cmap_int(i % 12)
    poly = MplPolygon(coords, closed=True, facecolor=color, edgecolor='red', linewidth=1.0, alpha=0.7)
    ax.add_patch(poly)

ax.set_aspect('equal')
ax.autoscale_view()
ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_title(f'Road Segments (roads: {len(road_segments)}, intersections: {len(intersection_segments)})')
plt.tight_layout()
plt.show()

In [ ]:
# Road Blockの可視化（要素ごとに色分け + from/to edge line）
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon

fig, ax = plt.subplots(1, 1, figsize=(14, 14))

# lineのtoken→座標列マッピング
line_map = {}
for l in map_data['line']:
    coords = [node_map[nt] for nt in l['node_tokens']]
    line_map[l['token']] = coords

# 背景としてdrivable areaを描画
for da in map_data['drivable_area']:
    for pt in da['polygon_tokens']:
        if pt in polygon_map:
            poly = MplPolygon(polygon_map[pt], closed=True, facecolor='lightgray', edgecolor='none', alpha=0.3)
            ax.add_patch(poly)

# Road Blockを色分けして描画
n_blocks = len(map_data['road_block'])
cmap = plt.cm.get_cmap('tab20', max(n_blocks, 1))

for i, rb in enumerate(map_data['road_block']):
    pt = rb['polygon_token']
    if pt not in polygon_map:
        continue
    coords = polygon_map[pt]
    color = cmap(i % 20)
    poly = MplPolygon(coords, closed=True, facecolor=color, edgecolor='black', linewidth=0.8, alpha=0.6)
    ax.add_patch(poly)

    # from_edge_line (青) と to_edge_line (緑) を描画
    for edge_key, edge_color, marker in [('from_edge_line_token', 'blue', 'o'), ('to_edge_line_token', 'green', 's')]:
        lt = rb.get(edge_key, '')
        if lt and lt in line_map:
            edge_coords = line_map[lt]
            xs = [c[0] for c in edge_coords]
            ys = [c[1] for c in edge_coords]
            ax.plot(xs, ys, color=edge_color, linewidth=2.0, marker=marker, markersize=4)

ax.set_aspect('equal')
ax.autoscale_view()
ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_title(f'Road Blocks ({n_blocks} blocks, from_edge=blue, to_edge=green)')
plt.tight_layout()
plt.show()

In [ ]:
# Laneの可視化（色分け）
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon

fig, ax = plt.subplots(1, 1, figsize=(14, 14))

# lineのtoken→座標列マッピング
line_map = {}
for l in map_data['line']:
    coords = [node_map[nt] for nt in l['node_tokens']]
    line_map[l['token']] = coords

# 背景としてdrivable areaを描画
for da in map_data['drivable_area']:
    for pt in da['polygon_tokens']:
        if pt in polygon_map:
            poly = MplPolygon(polygon_map[pt], closed=True, facecolor='lightgray', edgecolor='none', alpha=0.3)
            ax.add_patch(poly)

# Laneを色分けして描画
n_lanes = len(map_data['lane'])
cmap = plt.cm.get_cmap('tab20', max(n_lanes, 1))

for i, lane in enumerate(map_data['lane']):
    pt = lane['polygon_token']
    if pt not in polygon_map:
        continue
    coords = polygon_map[pt]
    color = cmap(i % 20)
    poly = MplPolygon(coords, closed=True, facecolor=color, edgecolor='black', linewidth=0.5, alpha=0.6)
    ax.add_patch(poly)

    # from_edge_line (青) と to_edge_line (緑) を描画
    for edge_key, edge_color, marker in [('from_edge_line_token', 'blue', 'o'), ('to_edge_line_token', 'green', 's')]:
        lt = lane.get(edge_key, '')
        if lt and lt in line_map:
            edge_coords = line_map[lt]
            xs = [c[0] for c in edge_coords]
            ys = [c[1] for c in edge_coords]
            ax.plot(xs, ys, color=edge_color, linewidth=2.0, marker=marker, markersize=4)

ax.set_aspect('equal')
ax.autoscale_view()
ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_title(f'Lanes ({n_lanes} lanes, from_edge=blue, to_edge=green)')
plt.tight_layout()
plt.show()

In [ ]:
# Road Dividerの可視化
import matplotlib.pyplot as plt

# lineのtoken→座標列マッピング
line_map = {}
for l in map_data['line']:
    coords = [node_map[nt] for nt in l['node_tokens']]
    line_map[l['token']] = coords

fig, ax = plt.subplots(1, 1, figsize=(14, 14))

# 背景としてdrivable areaを描画
for da in map_data['drivable_area']:
    for pt in da['polygon_tokens']:
        if pt in polygon_map:
            poly = MplPolygon(polygon_map[pt], closed=True, facecolor='lightgray', edgecolor='none', alpha=0.4)
            ax.add_patch(poly)

# Road Dividerを描画
for i, rd in enumerate(map_data['road_divider']):
    lt = rd['line_token']
    if lt not in line_map:
        continue
    coords = line_map[lt]
    xs = [c[0] for c in coords]
    ys = [c[1] for c in coords]
    ax.plot(xs, ys, color='red', linewidth=1.5, alpha=0.8)

ax.set_aspect('equal')
ax.autoscale_view()
ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_title(f'Road Dividers ({len(map_data["road_divider"])} lines)')
plt.tight_layout()
plt.show()

In [ ]:
# Lane Dividerの可視化
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(14, 14))

# 背景としてdrivable areaを描画
for da in map_data['drivable_area']:
    for pt in da['polygon_tokens']:
        if pt in polygon_map:
            poly = MplPolygon(polygon_map[pt], closed=True, facecolor='lightgray', edgecolor='none', alpha=0.4)
            ax.add_patch(poly)

# Lane Dividerを描画
for i, ld in enumerate(map_data['lane_divider']):
    lt = ld['line_token']
    if lt not in line_map:
        continue
    coords = line_map[lt]
    xs = [c[0] for c in coords]
    ys = [c[1] for c in coords]
    ax.plot(xs, ys, color='blue', linewidth=1.0, alpha=0.7)

ax.set_aspect('equal')
ax.autoscale_view()
ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_title(f'Lane Dividers ({len(map_data["lane_divider"])} lines)')
plt.tight_layout()
plt.show()

In [ ]:
# ped_crossingの可視化
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon

fig, ax = plt.subplots(1, 1, figsize=(14, 14))

# 背景としてdrivable areaを描画
for da in map_data['drivable_area']:
    for pt in da['polygon_tokens']:
        if pt in polygon_map:
            poly = MplPolygon(polygon_map[pt], closed=True, facecolor='lightgray', edgecolor='none', alpha=0.3)
            ax.add_patch(poly)

# Crosswalkを描画
n_crosswalks = len(map_data['ped_crossing'])
cmap = plt.cm.get_cmap('Set2', max(n_crosswalks, 1))

for i, pc in enumerate(map_data['ped_crossing']):
    pt = pc['polygon_token']
    if pt not in polygon_map:
        continue
    coords = polygon_map[pt]
    color = cmap(i % 8)
    poly = MplPolygon(coords, closed=True, facecolor=color, edgecolor='black', linewidth=1.0, alpha=0.7)
    ax.add_patch(poly)

ax.set_aspect('equal')
ax.autoscale_view()
ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_title(f'Crosswalks / Ped Crossings ({n_crosswalks} polygons)')
plt.tight_layout()
plt.show()

In [ ]:
# Walkwayの可視化
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon

fig, ax = plt.subplots(1, 1, figsize=(14, 14))

# 背景としてdrivable areaを描画
for da in map_data['drivable_area']:
    for pt in da['polygon_tokens']:
        if pt in polygon_map:
            poly = MplPolygon(polygon_map[pt], closed=True, facecolor='lightgray', edgecolor='none', alpha=0.3)
            ax.add_patch(poly)

# Walkwayを描画
n_walkways = len(map_data['walkway'])
cmap = plt.cm.get_cmap('Pastel1', max(n_walkways, 1))

for i, ww in enumerate(map_data['walkway']):
    pt = ww['polygon_token']
    if pt not in polygon_map:
        continue
    coords = polygon_map[pt]
    color = cmap(i % 9)
    poly = MplPolygon(coords, closed=True, facecolor=color, edgecolor='brown', linewidth=0.8, alpha=0.7)
    ax.add_patch(poly)

ax.set_aspect('equal')
ax.autoscale_view()
ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_title(f'Walkways ({n_walkways} polygons)')
plt.tight_layout()
plt.show()

In [ ]:
# Stop Lineの可視化
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon

fig, ax = plt.subplots(1, 1, figsize=(14, 14))

# 背景としてdrivable areaを描画
for da in map_data['drivable_area']:
    for pt in da['polygon_tokens']:
        if pt in polygon_map:
            poly = MplPolygon(polygon_map[pt], closed=True, facecolor='lightgray', edgecolor='none', alpha=0.3)
            ax.add_patch(poly)

# Stop Lineをタイプ別に色分けして描画
type_colors = {
    'PED_CROSSING': 'orange',
    'STOP_SIGN': 'red',
    'TRAFFIC_LIGHT': 'green',
    'TURN_STOP': 'purple',
    'YIELD': 'blue',
}
n_stop_lines = len(map_data['stop_line'])

for i, sl in enumerate(map_data['stop_line']):
    pt = sl['polygon_token']
    if pt not in polygon_map:
        continue
    coords = polygon_map[pt]
    sl_type = sl.get('stop_line_type', 'UNKNOWN')
    color = type_colors.get(sl_type, 'gray')
    poly = MplPolygon(coords, closed=True, facecolor=color, edgecolor='black', linewidth=1.0, alpha=0.7)
    ax.add_patch(poly)

# 凡例
from matplotlib.patches import Patch
legend_patches = [Patch(facecolor=c, edgecolor='black', label=t) for t, c in type_colors.items()]
ax.legend(handles=legend_patches, loc='upper right')

ax.set_aspect('equal')
ax.autoscale_view()
ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_title(f'Stop Lines ({n_stop_lines} polygons)')
plt.tight_layout()
plt.show()